# Exercise 3: RAG & Knowledge Integration

## Learning Objectives

In this exercise, you will:
- Understand the difference between tools (function calling) and RAG (knowledge retrieval)
- Configure and use Vertex AI RAG Engine for destination knowledge
- Create RAG-only agents with knowledge base access
- Test semantic retrieval with destination guides
- Combine function calling tools with RAG in a hybrid agent pattern

**Estimated time:** 40 minutes

## What is RAG (Retrieval-Augmented Generation)?

**⏱️ Estimated time: ~5 minutes**

<!-- INSTRUCTOR: This builds on Phase 2 - emphasize the complementary relationship -->

RAG gives your agent access to **private knowledge** - documents, guides, policies - that weren't in the LLM's training data. The agent retrieves relevant information from a knowledge base and uses it to generate accurate, grounded responses.

---

### How RAG Works

```
1. Documents indexed into RAG corpus (PDFs → chunks → embeddings)
          ↓
2. User asks: "What are visa requirements for Japan?"
          ↓
3. Agent searches corpus (semantic similarity)
          ↓
4. Top relevant chunks retrieved (with similarity scores)
          ↓
5. Agent uses chunks as context to answer question
```

---

### Tools vs RAG: The Decision Framework (Revisited)

Recall from Exercise 2 - you choose based on data characteristics:

```mermaid
flowchart TD
    A[User Query] --> B{Does this need\nREAL-TIME data?}
    B -->|YES| C[Use FUNCTION CALLING\nsearch_flights, search_hotels]
    B -->|NO| D{Is this STATIC\nKNOWLEDGE?}
    D -->|YES| E[Use RAG RETRIEVAL\nretrieve_destination_info]
    D -->|NO| F[Direct LLM\ngeneral knowledge]
```

| Question | Tool or RAG? | Why? |
|----------|--------------|------|
| "Find flights to Tokyo" | **TOOL** (search_flights) | Prices/availability change constantly |
| "What's the best time to visit Tokyo?" | **RAG** (retrieve_destination_info) | Static seasonal guide |
| "Book hotel in Shibuya under $200" | **TOOL** (search_hotels) | Real-time pricing |
| "What are visa requirements for Japan?" | **RAG** (retrieve_destination_info) | Static immigration policy |
| "What's the capital of Japan?" | **Neither** (LLM knows this) | General knowledge |

> 💡 **Key Insight:** Use **tools for dynamic data** that changes while user is talking. Use **RAG for static knowledge** in documents.

---

### Why RAG for Travel Guides?

**Perfect use case:**
- Destination guides rarely change (update quarterly, not hourly)
- Contains structured information (visa rules, attractions, cultural tips)
- Too much content to fit in LLM context window
- Needs semantic search ("safety tips" retrieves safety sections)

**In this workshop:**
- Pre-indexed corpus: 10 destination guides (Tokyo, Paris, NYC, Singapore, etc.)
- Each guide: visa requirements, attractions, weather, culture, transportation, food
- Layout-aware chunking: tables and lists preserved intact
- Document AI parser: understands document structure for better retrieval

## Setup Instructions

**⏱️ Estimated time: ~2 minutes**

This exercise assumes you have completed Exercise 1 and Exercise 2.

If you haven't completed setup yet, please go back to: [00-setup-verification.ipynb](00-setup-verification.ipynb)

In [ ]:
# @title Quick Setup (Run this cell first)
# This cell sets up your environment - you can collapse it after running

import os
import google.generativeai as genai
from getpass import getpass

# Step 1: Install Google ADK
!pip install -q google-adk==1.23.0
print("✓ google-adk 1.23.0 installed")

# Step 2: Configure API Key
api_key = getpass('Enter your Google AI API Key: ')
genai.configure(api_key=api_key)
os.environ['GOOGLE_API_KEY'] = api_key
print("✓ API Key configured")

print("\n🚀 Ready to integrate RAG knowledge!")

In [ ]:
# Import ADK components for agents and RAG
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part
from google.adk.tools.retrieval.vertex_ai_rag_retrieval import VertexAiRagRetrieval
from vertexai.preview import rag

# Create session service for conversation management
session_service = InMemorySessionService()
user_id = "workshop_participant"

print("✓ ADK components imported")
print("✓ RAG retrieval tools imported")
print("✓ Session service created")

## Exercise 3A: Explore Pre-Indexed RAG Corpus

**⏱️ Estimated time: ~5 minutes**

<!-- INSTRUCTOR: Show the corpus in the console if possible -->

The instructor has pre-created a RAG corpus with destination guide PDFs. This saves 15-20 minutes of setup time and lets us focus on **using** RAG, not infrastructure.

### What's in the Corpus?

**10 destination guides**, each with standardized sections:
1. Quick Facts (language, currency, emergency numbers)
2. Visa Requirements (by nationality)
3. Best Time to Visit (seasonal breakdown)
4. Top Attractions (table format with fees, hours, tips)
5. Neighborhoods & Districts
6. Food & Dining
7. Transportation
8. Cultural Tips & Customs
9. Safety & Health
10. Practical Information

**Destinations covered:**
- Tokyo, Japan
- Paris, France
- New York City, USA
- Singapore
- London, UK
- Rome, Italy
- Bangkok, Thailand
- Sydney, Australia
- Barcelona, Spain
- Dubai, UAE

### How the Guides Were Indexed

**Chunking strategy:**
- Chunk size: 1024 tokens (~800 words)
- Chunk overlap: 256 tokens (~200 words)
- Document AI layout parser enabled (preserves tables/lists)

**Why layout parsing matters:**
- WITHOUT: Table splits mid-row → broken data
- WITH: Full tables in single chunk → clean retrieval

Example:
```
| Attraction | Price | Hours |
|------------|-------|-------|
| Temple     | Free  | 6am   |
| Museum     | $15   | 9am   |
```
↑ This stays intact in one chunk with layout parser

In [ ]:
# Configure RAG Corpus ID
# The instructor will provide this during the workshop

# TODO: Get this from your instructor
# Format: projects/{project}/locations/{location}/ragCorpora/{corpus_id}
RAG_CORPUS_ID = "projects/your-workshop-project/locations/us-central1/ragCorpora/1234567890"

# Store in environment for later use
os.environ['RAG_CORPUS_ID'] = RAG_CORPUS_ID

print(f"✓ RAG Corpus ID configured")
print(f"  Corpus: {RAG_CORPUS_ID[:60]}...")
print("\n📚 Corpus contains 10 destination guides with ~15-25 chunks each")

### ✅ Checkpoint: Corpus Configured

<!-- INSTRUCTOR: Verify all participants have the corpus ID before proceeding -->

You should see:
- ✓ RAG Corpus ID configured
- Corpus path starting with `projects/...`

**If you see an error:**
- Make sure you replaced the placeholder corpus ID with the one from your instructor
- Check that the format matches: `projects/{project}/locations/{location}/ragCorpora/{id}`